# `find_nearest_point()`

`nematics3d.geometry.find_nearest_point()` finds the point in a finite point set that is closest to a query point in Euclidean distance.

**Important limitation:** if multiple candidate points are exactly the same minimum distance from the query point, `find_nearest_point()` currently returns only the first one in `coords`. The selected point can therefore depend on input row order in tie cases. This behavior is intentional for the current API, but a future design may provide a better way to represent multiple equally near points.

## What `find_nearest_point()` is for

Use `find_nearest_point()` when you have one query point and an array of candidate points and need the candidate with the smallest Euclidean distance to the query. The points may have any spatial dimension, provided the query point and every candidate point have the same dimension.

The function computes squared Euclidean distances, so it identifies the same nearest point as computing the distances themselves without taking unnecessary square roots.

## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** The following cell only imports `NumPy` and `find_nearest_point()`.

In [1]:
import numpy as np

from nematics3d.geometry import find_nearest_point

ImportError: cannot import name 'repr_field_line' from partially initialized module 'nematics3d.format' (most likely due to a circular import) (D:\Document\GitHub\Nematics3D\src\nematics3d\format.py)

## Minimal example

Here the query point is compared with four candidate points in two dimensions.

In [ ]:
query_point = np.array([1.0, 1.0])
points = np.array([
    [0.0, 0.0],
    [0.8, 1.2],
    [2.0, 1.0],
    [3.0, 3.0],
])

nearest_point = find_nearest_point(query_point, points)
print(nearest_point)

### What is returned

By default, the function returns a one-dimensional `NumPy` array containing the nearest point. For the example above, the result is `[0.8, 1.2]`.

The returned point is a copy, so modifying it does not modify the original `points` array.

## Arguments

The public signature is:

```python
find_nearest_point(query_pt, coords, is_return_idx=False)
```

| Argument | What it controls | Typical form |
| --- | --- | --- |
| `query_pt` | The point from which distances are measured. It must be a finite one-dimensional array-like object. | `np.array([x, y, z])` |
| `coords` | Candidate points. It must be a non-empty finite two-dimensional array-like object with one row per point and the same number of columns as `query_pt` has entries. | shape `(N, D)` |
| `is_return_idx=False` | Whether to return the row index of the nearest point together with the point itself. It must be a boolean. | `True` or `False` |

### Example: also return the point index

Set `is_return_idx=True` when you also need to locate the nearest point in the original candidate array.

In [ ]:
nearest_point, index = find_nearest_point(
    query_point,
    points,
    is_return_idx=True,
)

print("nearest point:", nearest_point)
print("index:", index)
print("points[index]:", points[index])

## Special examples

### Special case: dimensions other than 3D

The function is not restricted to three-dimensional geometry. The same calculation works in any dimension $D$ as long as `query_pt` has shape `(D,)` and `coords` has shape `(N, D)`.

In [ ]:
query_4d = np.array([0.0, 0.0, 0.0, 0.0])
points_4d = np.array([
    [1.0, 1.0, 1.0, 1.0],
    [0.1, 0.2, 0.1, 0.2],
    [2.0, 0.0, 0.0, 0.0],
])

print(find_nearest_point(query_4d, points_4d))

### Special case: equally near points

If multiple candidate points have exactly the same minimum distance, `find_nearest_point()` returns the first one in `coords`. This follows the first-minimum behavior of `numpy.argmin()`. Reordering equally near candidates can therefore change which point is returned.

In [ ]:
query = np.array([0.0, 0.0])
equidistant_points = np.array([
    [1.0, 0.0],
    [-1.0, 0.0],
])

point, index = find_nearest_point(
    query,
    equidistant_points,
    is_return_idx=True,
)
print(point, index)

## Details

For a query point $\mathbf{q}$ and candidate points $\mathbf{x}_i$, the function minimizes the squared Euclidean distance

$$d_i^2 = \lVert \mathbf{x}_i - \mathbf{q} \rVert^2.$$

Taking a square root is unnecessary because the square-root function is monotonic for non-negative values. The implementation therefore forms the coordinate differences and evaluates their row-wise squared norms directly.

All coordinates must be finite. Empty candidate sets, mismatched dimensions, `NaN`, and infinite coordinates are rejected explicitly because a nearest point would otherwise be undefined or misleading.

This implementation performs a direct $O(ND)$ scan for one query against $N$ points in $D$ dimensions. That is appropriate for isolated nearest-point queries. If the same large point set must be queried many times, a spatial search structure such as a k-d tree may be more appropriate because its construction cost can then be amortized over many queries.

## Summary

- `find_nearest_point()` finds the candidate with minimum Euclidean distance to one query point.
- `coords` has shape `(N, D)` and `query_pt` has shape `(D,)`; the function is not restricted to 3D.
- Set `is_return_idx=True` to also obtain the row index in `coords`.
- Exact ties are resolved by returning the first matching row, so tie results can depend on input order.
- Inputs must be non-empty, dimensionally compatible, and finite.